# Essai : le jeu de base, complet, `PAY_n` seuls, ou sans les `PAY_n`

**Problématique** : le comportement du client explique-t-il le défaut ?

**Question de l'essai** : sur le jeu de base (niveau 0 du nettoyage, sans aucune correction métier ni colonne créée), que porte chaque famille de colonnes ? Les codifications `PAY_1` à `PAY_6` sont l'étiquette posée par la banque ; les montants (factures, paiements, plafond) sont le comportement du client lui-même. Si les `PAY_n` seuls font aussi bien que les 23 variables, et que les montants sans eux font nettement moins bien, ce que les modèles trouvent tient surtout à l'étiquette de la banque, pas au comportement de paiement tel que les montants le décrivent. Trois jeux de variables sont comparés :

| Variante | Variables | Nombre |
|---|---|---|
| **Complet** | les 23 variables d'origine | 23 |
| **`PAY_n` seuls** | `PAY_1` à `PAY_6` | 6 |
| **Sans les `PAY_n`** | les 23 variables moins `PAY_1` à `PAY_6` : plafond, démographie, factures, paiements | 17 |

**Ce n'est pas un scénario** : c'est un essai à part. Rien n'est écrit dans le fichier de résultats ni dans le journal, aucun modèle n'est enregistré, aucune probabilité n'est enregistrée.

**Données** : `data/creditcard_pret_ingestion.csv`, produit par `02_01_nettoyage.ipynb` (niveau 0 : nettoyage structurel seul, catégories non documentées de `EDUCATION` et `MARRIAGE` regroupées). Ce sont les **30 000 clients** du fichier d'origine : les clients au contentieux sont présents, les `PAY_n` ne sont pas corrigés (faux retards, mois de transition), les factures négatives (avoirs), les paiements géants, les comptes inactifs et les plafonds atypiques sont gardés. Les scores ne se comparent donc pas directement à ceux des scénarios `ml_n` (autre population, autre taux de défaut).

⚠️ **Le test du ML est inclus** : ce fichier contient les clients de `data/ML/test_dataset.csv`, qui passent ici dans la validation croisée. C'est un choix assumé pour tester le jeu de base en entier. En conséquence, **aucune conclusion de cet essai ne doit servir à choisir le modèle final ou ses variables** : elle éclaire une hypothèse, elle ne décide pas du ML.

**Objectif** : détecter **le plus de défauts possible sans trop dégrader la précision**, et laisser chaque modèle aller au bout de sa capacité de détection, au lieu de le juger à un rappel fixé d'avance comme dans les scénarios.

**Méthode** : la boucle des scénarios (`methodologie_ML.md`), avec trois changements :
- validation croisée stratifiée à 5 plis, `random_state=42`, sur les 30 000 clients ;
- **4 modèles** : régression logistique, MLP, RandomForest, CatBoost (les meilleurs des scénarios) ;
- **optimisation automatique** des hyperparamètres par la boucle de GridSearch des scénarios (mêmes grilles de départ, mêmes corrections de grille, mêmes règles d'arrêt) ;
- **changement 1, le critère de la boucle** : la **précision moyenne** (aire sous la courbe précision-rappel) au lieu de la précision au rappel minimal de 60 %. Elle juge la précision du modèle sur **tous les rappels**, et pas en un seul point : elle récompense un modèle qui reste précis quand on va chercher beaucoup de défauts. Comme la précision, elle vaut le taux de défaut pour un tri au hasard ;
- **changement 2, le seuil de surapprentissage** : **7 %** d'écart relatif train - validation de la précision moyenne (5 % dans les scénarios), pour laisser les modèles aller plus loin ;
- **changement 3, le seuil de décision optimisé automatiquement** : chaque modèle signale des clients, du plus risqué au moins risqué, **tant que les clients ajoutés valent la peine d'être signalés** : au plus **1 bon client signalé pour 1 défaut attrapé, à la marge**. Le seuil retenu est celui qui maximise le **gain net** « défauts détectés − bons clients signalés » : chaque client ajouté rapporte +1 s'il est en défaut, coûte −1 sinon, donc le gain monte tant que les clients ajoutés sont en défaut au moins une fois sur deux (**précision marginale** de 50 %), puis redescend. Le rapport (`BONS_CLIENTS_PAR_DEFAUT`) est fixé à l'avance par le métier, pas calculé sur les données ; seul le seuil qui le réalise est calculé sur les données. Il est choisi par une validation croisée interne aux plis d'entraînement (`TunedThresholdClassifierCV`), puis jugé sur le pli de validation, qu'il n'a jamais vu. Chaque modèle a ainsi son propre seuil, MLP compris (le seuil se règle sur les probabilités, quel que soit le modèle).
  Deux règles ont été écartées. **Le F2 maximal** s'arrête quand la précision marginale tombe à F2 / 5 (environ 1 défaut pour 7 bons clients ici) : avec 22 % de défauts, il conduit à signaler la majorité des clients. **Un plancher sur la précision moyenne** des clients signalés laisse passer, à la marge, des clients bien moins souvent en défaut que le plancher ;
- comparaison des variantes, pli par pli, sur le **rappel au seuil optimisé** (combien de défauts chaque modèle détecte avec cette règle) et sur la **précision moyenne**, avec le verdict des scénarios (gain réel, perte, dans le bruit) ;
- **ce qui porte les prédictions** : importance par permutation sur les plis de validation (perte de précision moyenne quand une famille de colonnes, ou une colonne, est mélangée) ;
- comparaison avec les scénarios **sur les mêmes clients** : les probabilités hors pli du jeu de base, restreintes aux clients du train ML et découpées selon les plis des scénarios, face aux probabilités hors pli enregistrées par les scénarios (`data/ML/probas_hors_pli/`). Cette comparaison reste au **rappel minimal de 60 %**, la mesure des scénarios.

Le notebook est reproductible : graines fixées partout, mêmes plis pour les trois variantes, aucun fichier intermédiaire. Le relancer en entier redonne les mêmes résultats.

**Étapes** :
1. chargement du jeu de base et variables des trois variantes ;
2. pipeline, critère et boucle (repris des scénarios) ;
3. boucle d'optimisation, variante par variante ;
4. seuil de décision optimisé (au plus 1 bon client pour 1 défaut, à la marge) et comparaison des variantes, pli par pli ;
5. ce qui porte les prédictions : importance par permutation, par famille de colonnes et par colonne ;
6. comparaison avec le travail mené dans les scénarios (`ml_0`, `ml_4`), sur les mêmes clients.

## 1. Chargement du jeu de base et variables des trois variantes

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

original = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")   # niveau 0 du nettoyage (02_01_nettoyage.ipynb)
assert original["ID"].is_unique and not original.isna().any().any()
y_train = original["dpnm"].reset_index(drop=True)
print(f"Clients : {len(original)} ; colonnes : {original.shape[1]} (ID, 23 variables, dpnm)")
print(f"Défauts : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

Clients : 30000 ; colonnes : 25 (ID, 23 variables, dpnm)
Défauts : 6636 (22.12 %)


In [12]:
# ==============================================================================
# VARIABLES : les 23 variables d'origine, rangées selon leur encodage (celui des scénarios, méthode v8)
# ==============================================================================
PAY = [f"PAY_{n}" for n in range(1, 7)]

# PAY_n : nombres ordonnés (-2 à 8), mis à l'échelle (v6)
# PAY_AMTn : paiements très étirés, logarithme (v7)
# BILL_AMTn : factures, logarithme signé, car elles peuvent être négatives (v8)
# SEX, EDUCATION, MARRIAGE : catégories sans ordre, une colonne par catégorie (regroupées au niveau 0)
TYPES_ORIGINE = {
    "num": ["LIMIT_BAL", "AGE"] + PAY,
    "log": [f"PAY_AMT{n}" for n in range(1, 7)],
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "nom": ["SEX", "EDUCATION", "MARRIAGE"],
}

# Les trois variantes : colonnes gardées, chacune avec son encodage d'origine
VARIANTES = {
    "Complet": lambda c: True,
    "PAY_n seuls": lambda c: c in PAY,
    "Sans les PAY_n": lambda c: c not in PAY,
}
TYPES_VARIANTE = {v: {t: [c for c in cols if garder(c)] for t, cols in TYPES_ORIGINE.items()} for v, garder in VARIANTES.items()}
COLONNES_VARIANTE = {v: [c for cols in types.values() for c in cols] for v, types in TYPES_VARIANTE.items()}

# Contrôles : colonnes présentes, ni identifiant ni cible, les 23 variables couvertes par la variante complète
assert sorted(COLONNES_VARIANTE["Complet"]) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))
assert sorted(COLONNES_VARIANTE["PAY_n seuls"] + COLONNES_VARIANTE["Sans les PAY_n"]) == sorted(COLONNES_VARIANTE["Complet"])
assert (original[TYPES_ORIGINE["log"]] >= 0).all().all(), "Le logarithme ne s'applique qu'à des montants positifs ou nuls"

display(pd.DataFrame({
    "Variante": list(VARIANTES),
    "Nombre de variables": [len(COLONNES_VARIANTE[v]) for v in VARIANTES],
    "Variables": [", ".join(COLONNES_VARIANTE[v]) for v in VARIANTES],
}).set_index("Variante"))

,Nombre de variables,Variables
Variante,,
Complet,23,"LIMIT_BAL, AGE, PAY_1, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, PAY_AMT5, PAY_AMT6, BILL_AMT1, BILL_AMT2, BILL_AMT3, BILL_AMT4, BILL_AMT5, BILL_AMT6, SEX, EDUCATION, MARRIAGE"
PAY_n seuls,6,"PAY_1, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6"
Sans les PAY_n,17,"LIMIT_BAL, AGE, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, PAY_AMT5, PAY_AMT6, BILL_AMT1, BILL_AMT2, BILL_AMT3, BILL_AMT4, BILL_AMT5, BILL_AMT6, SEX, EDUCATION, MARRIAGE"


## 2. Pipeline, critère et boucle : repris des scénarios

Le pipeline et la boucle sont ceux des scénarios (`ml_13`, méthode v8), à quatre différences près : le **critère de la boucle**, la précision moyenne (`CRITERE`, au lieu de la précision au rappel minimal) ; le **seuil de surapprentissage à 7 %**, mesuré sur ce critère ; les **4 modèles** suivis ; un pipeline construit **pour chaque variante** (seules ses colonnes y entrent). Le plancher contre le sous-apprentissage ne change pas de sens : la précision moyenne d'un tri au hasard vaut le taux de défaut. Aucune valeur n'est remplacée : le jeu de base n'a pas de vide.

In [13]:
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)   # textes affichés en entier

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def preprocesseur(variante):
    # Une branche par type de colonne, seulement pour les types présents dans la variante
    return ColumnTransformer(transformers=[(nom, TRANSFORMATIONS[nom](), colonnes)
                                           for nom, colonnes in TYPES_VARIANTE[variante].items() if colonnes])

# Critère optimisé par la boucle : la précision moyenne (aire sous la courbe précision-rappel), sur tous les rappels
CRITERE = "average_precision"
scorers = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
}

# Rappel minimal des scénarios (60 %) : sert seulement à la comparaison avec les scénarios (section 6)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles de l'essai : les meilleurs des scénarios, avec leurs réglages de la première itération
MODELES_A_TESTER = ["LogisticRegression", "MLPClassifier", "RandomForest", "CatBoost"]

def modeles_de_base(variante):
    prep = preprocesseur(variante)
    tous = {
        "LogisticRegression": make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "MLPClassifier": make_pipeline(prep, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "RandomForest": make_pipeline(prep, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(prep, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

# ------------------------------------------------------------------------------
# Alertes et tableaux de scores (ceux des scénarios)
SEUIL_SURAPPRENTISSAGE = 0.07   # écart relatif train - val du critère : 7 % pour cet essai (5 % dans les scénarios) ; le double : surapprentissage fort
SEUIL_INSTABILITE = 0.02        # écart-type du critère sur les plis au-delà duquel on alerte (alerte seule, pas une contrainte)
METRIQUES = {"average_precision": "Précision moyenne", "roc_auc": "ROC AUC"}
LIBELLE_SCORE = "Précision des défauts prédits"
TAUX_DEFAUT = y_train.mean()    # précision, et précision moyenne, d'un tri au hasard

def apprend(score_val, std_val):
    # Plancher contre le sous-apprentissage : le critère doit dépasser le hasard (taux de défaut) de plus d'un écart-type
    return score_val - TAUX_DEFAUT > std_val

def alerte(ecart):
    if ecart > 2 * SEUIL_SURAPPRENTISSAGE:
        return "⚠️⚠️ SURAPPRENTISSAGE FORT"
    if ecart > SEUIL_SURAPPRENTISSAGE:
        return "⚠️ SURAPPRENTISSAGE"
    return "✅ OK"

def ligne_scores(nom, moy_val, std_val, moy_train):
    # Une ligne de tableau : chaque métrique en entraînement et en validation, le critère, son écart-type et l'alerte
    ligne = {"Modèle": nom}
    for cle, libelle in METRIQUES.items():
        ligne[f"{libelle} (train)"] = moy_train[cle]
        ligne[f"{libelle} (val)"] = moy_val[cle]
    ligne["Critère (val)"], ligne["Critère (train)"] = moy_val[CRITERE], moy_train[CRITERE]
    ligne["Écart-type critère (val)"] = std_val
    ligne["Écart train - val"] = moy_train[CRITERE] - moy_val[CRITERE]
    ligne["Écart relatif"] = ligne["Écart train - val"] / moy_val[CRITERE]
    ligne["Apprend"] = apprend(moy_val[CRITERE], std_val)
    ligne["Alerte"] = alerte(ligne["Écart relatif"]) if ligne["Apprend"] else "⚠️ N'APPREND RIEN (niveau du hasard)"
    if std_val > SEUIL_INSTABILITE:
        ligne["Alerte"] += " ; ⚠️ INSTABLE"
    return ligne

def pct(serie):
    return (serie * 100).round(1)

def parametres(params):
    # Paramètres sur une ligne : « depth 4, 5, 6 ; l2_leaf_reg 15 »
    return " ; ".join(f"{p} {', '.join(map(str, v)) if isinstance(v, list) else v}" for p, v in params.items())

def montrer(df, titre):
    display(Markdown(f"**{titre}**"))
    display(df.set_index(df.columns[0]))

In [14]:
from itertools import product
from sklearn.model_selection import GridSearchCV

TOURS_SECURITE = 15   # garde-fou contre une boucle sans fin : la boucle s'arrête d'elle-même avant ; s'il est atteint, revoir les grilles ou les bornes

# Grilles du premier tour : celles des scénarios
grilles = {
    "LogisticRegression": {"logisticregression__C": [0.01, 0.1, 1.0, 10.0]},
    "MLPClassifier": {"mlpclassifier__hidden_layer_sizes": [(32, 16), (64, 32), (32,)], "mlpclassifier__alpha": [0.01, 0.1, 1.0]},
    "RandomForest": {"randomforestclassifier__max_depth": [6, 8, 10], "randomforestclassifier__min_samples_leaf": [20, 50, 100], "randomforestclassifier__n_estimators": [200, 300]},
    "CatBoost": {"catboostclassifier__depth": [4, 6], "catboostclassifier__l2_leaf_reg": [5, 10, 15], "catboostclassifier__learning_rate": [0.03, 0.05],
                 "catboostclassifier__iterations": [1000]},   # 1 000 arbres : valeur par défaut, réduite par la boucle en cas de surapprentissage
}

# Comment faire évoluer chaque paramètre numérique d'un tour à l'autre (ceux des scénarios) :
# - simplifier : sens qui réduit le surapprentissage (+1 = augmenter, -1 = diminuer) ;
# - pas : écart entre deux valeurs (« x » = multiplier, « + » = ajouter) ;
# - bornes : valeurs minimale et maximale autorisées.
# Les paramètres absents de ce tableau (architecture du MLP, nombre d'arbres du RandomForest) sont fixés à leur meilleure valeur après le premier tour.
REGLAGES_PARAMETRES = {
    "logisticregression__C":                    {"simplifier": -1, "pas": ("x", 3),   "bornes": (1e-4, 1e4),  "entier": False},
    "mlpclassifier__alpha":                     {"simplifier": +1, "pas": ("x", 3),   "bornes": (1e-5, 100),  "entier": False},
    "randomforestclassifier__max_depth":        {"simplifier": -1, "pas": ("+", 2),   "bornes": (2, 30),      "entier": True},
    "randomforestclassifier__min_samples_leaf": {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 2000),    "entier": True},
    "catboostclassifier__depth":                {"simplifier": -1, "pas": ("+", 1),   "bornes": (2, 10),      "entier": True},
    "catboostclassifier__l2_leaf_reg":          {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 200),     "entier": False},
    "catboostclassifier__learning_rate":        {"simplifier": -1, "pas": ("x", 1.5), "bornes": (0.003, 0.3), "entier": False},
    "catboostclassifier__iterations":           {"simplifier": -1, "pas": ("x", 1.5), "bornes": (100, 3000),  "entier": True},
}

def nom_court(parametre):
    return parametre.split("__")[-1]

def arrondir(valeur, entier):
    return int(round(valeur)) if entier else float(f"{valeur:.3g}")

def voisin(valeur, reglage, sens):
    # Valeur suivante dans un sens (+1 = plus grande, -1 = plus petite), dans les bornes
    mode, pas = reglage["pas"]
    if mode == "x":
        nouvelle = valeur * pas if sens > 0 else valeur / pas
    else:
        nouvelle = valeur + pas if sens > 0 else valeur - pas
    bas, haut = reglage["bornes"]
    nouvelle = arrondir(min(max(nouvelle, bas), haut), reglage["entier"])
    return None if nouvelle == valeur else nouvelle

def choisir_index(r):
    # Choix sous contrainte, parmi les réglages qui battent le hasard : le meilleur critère sans surapprentissage ;
    # sinon le moins surappris ; si aucun ne bat le hasard, le meilleur score
    val, std = r[f"mean_test_{CRITERE}"], r[f"std_test_{CRITERE}"]
    ecarts = (r[f"mean_train_{CRITERE}"] - val) / val   # écarts relatifs
    utiles = apprend(val, std)
    sous_seuil = np.where(utiles & (ecarts <= SEUIL_SURAPPRENTISSAGE))[0]
    if len(sous_seuil):
        return int(sous_seuil[np.argmax(val[sous_seuil])])
    if utiles.any():
        idx = np.where(utiles)[0]
        return int(idx[np.argmin(ecarts[idx])])
    return int(np.argmax(val))

def bords_de_grille(grille, meilleurs):
    # Paramètres numériques dont la meilleure valeur est la plus petite ou la plus grande de la grille (+1 = maximum, -1 = minimum)
    bords = {}
    for p, valeurs in grille.items():
        if p in REGLAGES_PARAMETRES and len(valeurs) > 1:
            if meilleurs[p] == max(valeurs):
                bords[p] = +1
            elif meilleurs[p] == min(valeurs):
                bords[p] = -1
    return bords

def grille_suivante(grille, meilleurs, bords, surapprend):
    # Grille du tour suivant et explication des corrections ; None si plus rien ne peut bouger
    nouvelle, actions = {}, []
    for p, valeurs in grille.items():
        v = meilleurs[p]
        reglage = REGLAGES_PARAMETRES.get(p)
        if reglage is None:
            nouvelle[p] = [v]                            # catégorie ou paramètre sans effet : fixé
            continue
        sens = None
        if surapprend and reglage["simplifier"] != 0:
            sens = reglage["simplifier"]                 # pousser vers un modèle plus simple
            raison = "surapprentissage"
        elif p in bords:
            sens = bords[p]                              # prolonger la grille du côté du bord
            raison = "bord de grille"
        if sens is None:
            nouvelle[p] = [v]
            continue
        v1 = voisin(v, reglage, sens)
        v2 = voisin(v1, reglage, sens) if v1 is not None else None
        valeurs_tour = [x for x in (v, v1, v2) if x is not None]
        nouvelle[p] = sorted(set(valeurs_tour))
        if len(nouvelle[p]) > 1:
            actions.append(f"{nom_court(p)} {v} → {nouvelle[p]} ({raison})")
    if not actions:
        return None, ["bornes atteintes : la grille ne peut plus bouger"]
    return nouvelle, actions

def lancer_grille(nom, modele, grille, X):
    grid = GridSearchCV(modele, grille, cv=cv, scoring=scorers, refit=choisir_index, return_train_score=True, n_jobs=-1)
    grid.fit(X, y_train)
    r, i = grid.cv_results_, grid.best_index_
    ligne = ligne_scores(nom,
                         {k: r[f"mean_test_{k}"][i] for k in METRIQUES},
                         r[f"std_test_{CRITERE}"][i],
                         {k: r[f"mean_train_{k}"][i] for k in METRIQUES})
    ligne["Meilleurs paramètres"] = {nom_court(k): v for k, v in grid.best_params_.items()}
    return grid, ligne

def cle_reglage(params):
    # Identifiant d'une combinaison de paramètres, pour savoir si elle a déjà été testée
    return str(sorted(params.items()))

def reglages_nouveaux(grille, deja_testes):
    # Combinaisons de la grille qui n'ont encore été testées à aucun tour
    combinaisons = [dict(zip(grille, valeurs)) for valeurs in product(*grille.values())]
    return [c for c in combinaisons if cle_reglage(c) not in deja_testes]

def choisir_tour(tours):
    # Version retenue parmi les tours joués : d'abord ceux qui battent le hasard et sans surapprentissage
    # (sinon ceux qui battent le hasard, sinon tous) ; parmi eux, ceux à moins d'un écart-type du meilleur score
    # (à égalité avec lui) ; et parmi ces derniers, celui qui surapprend le moins
    for garder in (lambda l: l["Apprend"] and l["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE, lambda l: l["Apprend"], lambda l: True):
        groupe = [t for t in tours if garder(t[1])]
        if groupe:
            break
    meilleur = max(groupe, key=lambda t: t[1]["Critère (val)"])[1]
    egaux = [t for t in groupe if meilleur["Critère (val)"] - t[1]["Critère (val)"] <= meilleur["Écart-type critère (val)"]]
    return min(egaux, key=lambda t: t[1]["Écart relatif"])

def boucle(variante):
    # Boucle des scénarios, pour un jeu de variables : version retenue de chaque modèle, lignes de scores et historique
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    best_models, lignes_retenues, historique = {}, [], []
    for nom, modele in modeles_de_base(variante).items():
        grille, retenu, fin, deja_testes, reglage_precedent, tours = grilles[nom], None, None, set(), None, []
        for tour in range(1, TOURS_SECURITE + 1):
            grid, ligne = lancer_grille(nom, modele, grille, X)
            ligne["Tour"] = tour
            deja_testes |= {cle_reglage(c) for c in grid.cv_results_["params"]}
            surapprend = ligne["Écart relatif"] > SEUIL_SURAPPRENTISSAGE
            bords = bords_de_grille(grille, grid.best_params_)
            precedent = retenu
            tours.append((grid, ligne))
            retenu = choisir_tour(tours)
            grille_tour = {nom_court(p): v for p, v in grille.items()}
            reglage = ligne["Meilleurs paramètres"]
            # Stagnation : seulement entre deux tours sans surapprentissage
            stagne = (precedent is not None and not surapprend
                      and precedent[1]["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE
                      and ligne["Critère (val)"] - precedent[1]["Critère (val)"] <= precedent[1]["Écart-type critère (val)"])
            if not ligne["Apprend"]:
                fin = "sous-apprentissage : le réglage choisi ne bat plus le hasard"
            elif not surapprend and not bords:
                fin = "optimisé : ni surapprentissage ni paramètre au bord"
            elif reglage == reglage_precedent:
                fin = "convergé : le réglage choisi ne change plus"
            elif stagne:
                fin = "stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
            elif tour == TOURS_SECURITE:
                fin = "⚠️ garde-fou atteint : grilles ou bornes à revoir"
            else:
                grille, actions = grille_suivante(grille, grid.best_params_, bords, surapprend)
                if grille is None:
                    fin = actions[0]
                elif not reglages_nouveaux(grille, deja_testes):
                    fin = "convergé : la grille suivante ne contient aucun réglage nouveau"
            reglage_precedent = reglage
            historique.append({
                "Modèle": nom, "Tour": tour, "Grille testée": grille_tour, "Réglage choisi": reglage,
                "Critère (val)": ligne["Critère (val)"], "Écart-type": ligne["Écart-type critère (val)"],
                "Écart relatif": ligne["Écart relatif"], "Alerte": ligne["Alerte"],
                "Suite": fin if fin else "; ".join(actions),
            })
            if fin:
                break
        best_models[nom] = retenu[0].best_estimator_
        lignes_retenues.append(dict(retenu[1], **{"Arrêt": fin, "Tours joués": tour}))
        print(f"  {nom} : {tour} tour(s), {fin} ; version retenue : tour {retenu[1]['Tour']}")
    return best_models, pd.DataFrame(lignes_retenues), pd.DataFrame(historique)

## 3. Boucle d'optimisation, variante par variante

Pour chaque variante, la boucle règle les 4 modèles l'un après l'autre. Une ligne s'affiche pendant le calcul pour chaque modèle (nombre de tours, raison de l'arrêt, tour retenu). Puis, pour chaque variante :
- **Version retenue de chaque modèle** : la précision moyenne en validation (le critère), son écart-type, l'entraînement face à la validation, l'écart relatif (au-delà de 7 % : ⚠️ ; de 14 % : ⚠️⚠️), le ROC AUC en contrôle, et le réglage retenu ;
- **Historique de la boucle** : une ligne par tour et par modèle, avec la grille testée, le réglage choisi et la correction qui suit (ou la raison de l'arrêt).

Le calcul est long : la boucle tourne trois fois, sur 30 000 clients.

In [15]:
import time

resultats = {}
for variante in VARIANTES:
    debut = time.time()
    print(f"Variante « {variante} » ({len(COLONNES_VARIANTE[variante])} variables)")
    resultats[variante] = boucle(variante)
    print(f"  durée : {(time.time() - debut) / 60:.1f} min")

for variante, (_, df_retenus, df_historique) in resultats.items():
    display(Markdown(f"### Variante « {variante} »"))
    df = df_retenus.sort_values("Critère (val)", ascending=False)
    montrer(pd.DataFrame({
        "Modèle": df["Modèle"],
        "Précision moyenne val (%)": pct(df["Critère (val)"]),
        "Écart-type (pts)": pct(df["Écart-type critère (val)"]),
        "Précision moyenne train (%)": pct(df["Critère (train)"]),
        "Écart relatif (%)": pct(df["Écart relatif"]),
        "ROC AUC val": df["ROC AUC (val)"].round(3),
        "ROC AUC train": df["ROC AUC (train)"].round(3),
        "Alerte": df["Alerte"],
        "Tours joués": df["Tours joués"],
        "Tour retenu": df["Tour"],
        "Raison de l'arrêt": df["Arrêt"],
        "Réglage retenu": df["Meilleurs paramètres"].map(parametres),
    }), "Version retenue de chaque modèle (validation)")
    montrer(pd.DataFrame({
        "Modèle": df_historique["Modèle"],
        "Tour": df_historique["Tour"],
        "Grille testée": df_historique["Grille testée"].map(parametres),
        "Réglage choisi": df_historique["Réglage choisi"].map(parametres),
        "Précision moyenne val (%)": pct(df_historique["Critère (val)"]),
        "Écart relatif (%)": pct(df_historique["Écart relatif"]),
        "Alerte": df_historique["Alerte"],
        "Suite": df_historique["Suite"],
    }), "Historique de la boucle")

Variante « Complet » (23 variables)
  LogisticRegression : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  MLPClassifier : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  RandomForest : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  CatBoost : 3 tour(s), stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage ; version retenue : tour 2
  durée : 6.7 min
Variante « PAY_n seuls » (6 variables)
  LogisticRegression : 2 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 2
  MLPClassifier : 2 tour(s), stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage ; version retenue : tour 1
  RandomForest : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  CatBoost : 2 tour(s), stagnation : gain plus petit que l'écart-type, entre deux tours sans sur

### Variante « Complet »

**Version retenue de chaque modèle (validation)**

,Précision moyenne val (%),Écart-type (pts),Précision moyenne train (%),Écart relatif (%),ROC AUC val,ROC AUC train,Alerte,Tours joués,Tour retenu,Raison de l'arrêt,Réglage retenu
Modèle,,,,,,,,,,,
CatBoost,55.6,0.7,58.0,4.3,0.784,0.802,✅ OK,3,2,"stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage",depth 4 ; iterations 667 ; l2_leaf_reg 33.8 ; learning_rate 0.02
RandomForest,55.1,0.7,58.8,6.8,0.780,0.802,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,max_depth 8 ; min_samples_leaf 50 ; n_estimators 300
MLPClassifier,54.0,0.7,56.4,4.4,0.775,0.790,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,"alpha 0.1 ; hidden_layer_sizes (64, 32)"
LogisticRegression,49.9,0.4,50.1,0.4,0.749,0.751,✅ OK,2,1,convergé : le réglage choisi ne change plus,C 10.0


**Historique de la boucle**

...

### Variante « Sans les PAY_n »

**Version retenue de chaque modèle (validation)**

,Précision moyenne val (%),Écart-type (pts),Précision moyenne train (%),Écart relatif (%),ROC AUC val,ROC AUC train,Alerte,Tours joués,Tour retenu,Raison de l'arrêt,Réglage retenu
Modèle,,,,,,,,,,,
CatBoost,45.4,0.5,48.0,5.5,0.735,0.758,✅ OK,3,2,convergé : le réglage choisi ne change plus,depth 4 ; iterations 445 ; l2_leaf_reg 22.5 ; learning_rate 0.03
MLPClassifier,44.5,0.5,47.4,6.5,0.728,0.746,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,"alpha 0.1 ; hidden_layer_sizes (64, 32)"
RandomForest,43.8,0.7,46.7,6.7,0.723,0.744,✅ OK,2,1,convergé : le réglage choisi ne change plus,max_depth 6 ; min_samples_leaf 100 ; n_estimators 200
LogisticRegression,43.2,0.9,43.3,0.3,0.715,0.717,✅ OK,2,1,optimisé : ni surapprentissage ni paramètre au bord,C 10.0


**Historique de la boucle**

,Tour,Grille testée,Réglage choisi,Précision moyenne val (%),Écart relatif (%),Alerte,Suite
Modèle,,,,,,,
LogisticRegression,1,"C 0.01, 0.1, 1.0, 10.0",C 10.0,43.2,0.3,✅ OK,"C 10.0 → [10.0, 30.0, 90.0] (bord de grille)"
LogisticRegression,2,"C 10.0, 30.0, 90.0",C 30.0,43.2,0.3,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
MLPClassifier,1,"hidden_layer_sizes (32, 16), (64, 32), (32,) ; alpha 0.01, 0.1, 1.0","alpha 0.1 ; hidden_layer_sizes (64, 32)",44.5,6.5,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
RandomForest,1,"max_depth 6, 8, 10 ; min_samples_leaf 20, 50, 100 ; n_estimators 200, 300",max_depth 6 ; min_samples_leaf 100 ; n_estimators 200,43.8,6.7,✅ OK,"max_depth 6 → [2, 4, 6] (bord de grille); min_samples_leaf 100 → [100, 150, 225] (bord de grille)"
RandomForest,2,"max_depth 2, 4, 6 ; min_samples_leaf 100, 150, 225 ; n_estimators 200",max_depth 6 ; min_samples_leaf 100 ; n_estimators 200,43.8,6.7,✅ OK,convergé : le réglage choisi ne change plus
CatBoost,1,"depth 4, 6 ; l2_leaf_reg 5, 10, 15 ; learning_rate 0.03, 0.05 ; iterations 1000",depth 4 ; iterations 1000 ; l2_leaf_reg 15 ; learning_rate 0.03,45.5,12.6,⚠️ SURAPPRENTISSAGE,"depth 4 → [2, 3, 4] (surapprentissage); l2_leaf_reg 15 → [15, 22.5, 33.8] (surapprentissage); learning_rate 0.03 → [0.0133, 0.02, 0.03] (surapprentissage); iterations 1000 → [445, 667, 1000] (surapprentissage)"
CatBoost,2,"depth 2, 3, 4 ; l2_leaf_reg 15, 22.5, 33.8 ; learning_rate 0.0133, 0.02, 0.03 ; iterations 445, 667, 1000",depth 4 ; iterations 445 ; l2_leaf_reg 22.5 ; learning_rate 0.03,45.4,5.5,✅ OK,"depth 4 → [4, 5, 6] (bord de grille); learning_rate 0.03 → [0.03, 0.045, 0.0675] (bord de grille); iterations 445 → [198, 297, 445] (bord de grille)"
CatBoost,3,"depth 4, 5, 6 ; l2_leaf_reg 22.5 ; learning_rate 0.03, 0.045, 0.0675 ; iterations 198, 297, 445",depth 4 ; iterations 445 ; l2_leaf_reg 22.5 ; learning_rate 0.03,45.4,5.5,✅ OK,convergé : le réglage choisi ne change plus


**Lecture** : une section par variante. Le taux de défaut du jeu de base est la précision moyenne d'un tri au hasard : une précision moyenne proche de lui veut dire que le modèle ne trouve presque rien. Un modèle qui reste en surapprentissage malgré tous ses tours est signalé (⚠️) : sa grille de départ ou ses bornes sont peut-être à revoir pour ces données.

## 4. Seuil de décision optimisé et comparaison des variantes, pli par pli

**Seuil optimisé automatiquement** : sur chaque pli, la version retenue de chaque modèle est réentraînée sur les 4 autres plis, enveloppée dans `TunedThresholdClassifierCV`. Celui-ci découpe à son tour ces 4 plis en 5 plis internes et essaie une série de seuils sur les prédictions internes. Il garde celui qui maximise le **gain net** « défauts détectés − `BONS_CLIENTS_PAR_DEFAUT` × bons clients signalés », puis réentraîne le modèle sur les 4 plis. Le pli de validation n'intervient ni dans les réglages du modèle ni dans le choix du seuil : il juge seulement le résultat. Chaque client reçoit une probabilité hors pli et une décision (signalé ou non) prise au seuil de son pli. Les modèles entraînés sur chaque pli sont gardés en mémoire pour la section 5.

**Contrôle de la règle** : la précision **marginale** se mesure sur les clients signalés les plus proches du seuil (les 10 % de clients signalés qui ont les probabilités les plus basses). Elle doit tourner autour de 1 / (1 + `BONS_CLIENTS_PAR_DEFAUT`), soit 50 % : c'est la part de défauts parmi les derniers clients que la règle a jugé bon d'ajouter.

Trois tableaux :
- **Détection au seuil optimisé** : pour chaque modèle et chaque variante, le seuil choisi (moyenne et étendue sur les 5 plis), le **rappel** (part des défauts détectés), la **précision des défauts prédits** (sur tous les clients signalés), la **précision marginale** (sur les 10 % de signalés les plus proches du seuil), la part des clients signalés et le F1 (pour information), en moyenne sur les plis avec leur écart-type ; puis la précision moyenne et le ROC AUC, qui ne dépendent d'aucun seuil ;
- **Comparaison avec la variante complète** : pour chaque modèle, puis en meilleur contre meilleur, le gain moyen sur la variante complète, son écart-type et les plis gagnés, pour le rappel au seuil optimisé (combien de défauts sont détectés avec la règle) et pour la précision moyenne (la capacité de détection, tous seuils confondus). Verdict des scénarios : « gain réel » si le gain moyen dépasse l'écart-type des gains, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [16]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, fbeta_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TunedThresholdClassifierCV

y = y_train.to_numpy()
plis = list(cv.split(original, y_train))   # les mêmes 5 plis que la boucle
cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # plis internes, pour le choix du seuil

# Règle fixée à l'avance par le métier : au plus 1 bon client signalé pour 1 défaut attrapé, à la marge
BONS_CLIENTS_PAR_DEFAUT = 1
PRECISION_MARGINALE_VISEE = 1 / (1 + BONS_CLIENTS_PAR_DEFAUT)   # 50 % : précision des derniers clients ajoutés au seuil retenu
PART_MARGE = 0.10   # contrôle : précision mesurée sur les 10 % de clients signalés les plus proches du seuil

def gain_net(y_vrai, y_pred):
    # Défauts détectés moins les bons clients signalés (pondérés), rapporté au nombre de clients :
    # maximal au seuil où les clients ajoutés ne sont plus en défaut qu'avec la précision marginale visée
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    detectes = (y_pred & (y_vrai == 1)).sum()
    fausses_alertes = (y_pred & (y_vrai == 0)).sum()
    return (detectes - BONS_CLIENTS_PAR_DEFAUT * fausses_alertes) / len(y_vrai)

SCORE_SEUIL = make_scorer(gain_net)

def precision_marginale(y_vrai, proba, signales):
    # Part de défauts parmi les PART_MARGE des clients signalés qui ont les probabilités les plus basses (les plus proches du seuil)
    idx = np.where(signales)[0]
    if len(idx) == 0:
        return np.nan
    derniers = idx[np.argsort(proba[idx])[:max(1, int(np.ceil(PART_MARGE * len(idx))))]]
    return y_vrai[derniers].mean()

def f2_au_rappel_minimal(y_vrai, proba):
    # Mesure des scénarios (section 6) : F2 au seuil qui détecte au moins le rappel minimal
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

probas, mesures, modeles_plis = {}, {}, {}
for variante, (best_models, _, _) in resultats.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for nom, modele in best_models.items():
        p, entraines = np.zeros(len(y)), []
        m_plis = {"Seuil": [], "Rappel": [], "Précision": [], "Précision marginale": [], "Signalés": [], "F1": [], "Précision moyenne": [], "ROC AUC": []}
        for entrainement, validation in plis:
            seuil_auto = TunedThresholdClassifierCV(clone(modele), scoring=SCORE_SEUIL, cv=cv_interne, n_jobs=-1)
            seuil_auto.fit(X.iloc[entrainement], y[entrainement])
            p[validation] = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
            decision = seuil_auto.predict(X.iloc[validation]).astype(bool)
            y_val = y[validation]
            m_plis["Seuil"].append(seuil_auto.best_threshold_)
            m_plis["Rappel"].append(recall_score(y_val, decision))
            m_plis["Précision"].append(precision_score(y_val, decision, zero_division=0))
            m_plis["Précision marginale"].append(precision_marginale(y_val, p[validation], decision))
            m_plis["Signalés"].append(decision.mean())
            m_plis["F1"].append(fbeta_score(y_val, decision, beta=1))
            m_plis["Précision moyenne"].append(average_precision_score(y_val, p[validation]))
            m_plis["ROC AUC"].append(roc_auc_score(y_val, p[validation]))
            entraines.append(seuil_auto)
        probas[(variante, nom)], mesures[(variante, nom)], modeles_plis[(variante, nom)] = p, m_plis, entraines
        print(f"{variante}, {nom} : seuil {np.mean(m_plis['Seuil']):.3f}, rappel {np.mean(m_plis['Rappel']) * 100:.1f} %, "
              f"précision {np.mean(m_plis['Précision']) * 100:.1f} %, précision marginale {np.nanmean(m_plis['Précision marginale']) * 100:.1f} %")

def moy_et(valeurs, en_pct=False):
    facteur = 100 if en_pct else 1
    return f"{np.mean(valeurs) * facteur:.{1 if en_pct else 3}f} ± {np.std(valeurs) * facteur:.{1 if en_pct else 3}f}"

lignes = []
for (variante, nom), m in mesures.items():
    lignes.append({
        "Variante": variante, "Modèle": nom,
        "Seuil (moyenne)": round(np.mean(m["Seuil"]), 3),
        "Seuil (min - max)": f"{min(m['Seuil']):.3f} - {max(m['Seuil']):.3f}",
        "Rappel (%)": moy_et(m["Rappel"], True),
        f"{LIBELLE_SCORE} (%)": moy_et(m["Précision"], True),
        f"Précision marginale, {PART_MARGE:.0%} les plus proches du seuil (%)": moy_et(m["Précision marginale"], True),
        "Clients signalés (%)": moy_et(m["Signalés"], True),
        "F1": moy_et(m["F1"]),
        "Précision moyenne": moy_et(m["Précision moyenne"]),
        "ROC AUC": moy_et(m["ROC AUC"]),
    })
display(Markdown(f"**Détection au seuil optimisé (au plus {BONS_CLIENTS_PAR_DEFAUT} bon(s) client(s) pour 1 défaut à la marge, "
                 f"précision marginale visée {PRECISION_MARGINALE_VISEE:.0%}), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : {TAUX_DEFAUT * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

Complet, LogisticRegression : seuil 0.747, rappel 30.0 %, précision 62.4 %, précision marginale 48.1 %
Complet, MLPClassifier : seuil 0.516, rappel 35.5 %, précision 66.4 %, précision marginale 49.5 %
Complet, RandomForest : seuil 0.716, rappel 38.1 %, précision 63.9 %, précision marginale 49.8 %
Complet, CatBoost : seuil 0.771, rappel 37.3 %, précision 66.7 %, précision marginale 51.4 %
PAY_n seuls, LogisticRegression : seuil 0.683, rappel 37.8 %, précision 66.6 %, précision marginale 54.2 %
PAY_n seuls, MLPClassifier : seuil 0.511, rappel 35.4 %, précision 68.1 %, précision marginale 55.5 %
PAY_n seuls, RandomForest : seuil 0.786, rappel 35.5 %, précision 68.6 %, précision marginale 60.3 %
PAY_n seuls, CatBoost : seuil 0.777, rappel 36.2 %, précision 67.4 %, précision marginale 50.8 %
Sans les PAY_n, LogisticRegression : seuil 0.751, rappel 15.9 %, précision 57.0 %, précision marginale 50.0 %
Sans les PAY_n, MLPClassifier : seuil 0.489, rappel 22.1 %, précision 56.5 %, précision marg

**Détection au seuil optimisé (au plus 1 bon(s) client(s) pour 1 défaut à la marge, précision marginale visée 50%), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : 22.1 %)

Seuil (moyenne) Seuil (min - max)  Rappel (%) Précision des défauts prédits (%) Précision marginale, 10% les plus proches du seuil (%) Clients signalés (%)             F1 Précision moyenne        ROC AUC
Variante       Modèle                                                                                                                                                                                                                         
Complet        LogisticRegression            0.747     0.733 - 0.753  30.0 ± 2.3                        62.4 ± 1.4                                             48.1 ± 6.0           10.6 ± 1.0  0.404 ± 0.019     0.499 ± 0.004  0.749 ± 0.005
               MLPClassifier                 0.516     0.482 - 0.544  35.5 ± 1.9                        66.4 ± 2.5                                             49.5 ± 6.1           11.8 ± 0.9  0.462 ± 0.015     0.540 ± 0.007  0.775 ± 0.005
               RandomForest                  0.716     0.672 - 0.732  38.1 ± 3.5                        63.9 ± 0.9                                             49.8 ± 4.7           13.2 ± 1.4  0.476 ± 0.025     0.551 ± 0.007  0.780 ± 0.006
               CatBoost                      0.771     0.751 - 0.789  37.3 ± 1.7                        66.7 ± 1.1                                             51.4 ± 5.3           12.4 ± 0.6  0.478 ± 0.014     0.556 ± 0.007  0.784 ± 0.006
PAY_n seuls    LogisticRegression            0.683     0.676 - 0.689  37.8 ± 0.6                        66.6 ± 2.1                                             54.2 ± 6.7           12.6 ± 0.3  0.482 ± 0.009     0.486 ± 0.008  0.705 ± 0.006
               MLPClassifier                 0.511     0.478 - 0.550  35.4 ± 1.6                        68.1 ± 2.4                                             55.5 ± 5.3           11.5 ± 0.8  0.465 ± 0.011     0.518 ± 0.004  0.742 ± 0.009
               RandomForest                  0.786     0.784 - 0.787  35.5 ± 1.2                        68.6 ± 1.9                                             60.3 ± 4.5           11.5 ± 0.1  0.468 ± 0.014     0.531 ± 0.007  0.749 ± 0.006
               CatBoost                      0.777     0.764 - 0.787  36.2 ± 1.3                        67.4 ± 1.7                                             50.8 ± 6.1           11.9 ± 0.2  0.471 ± 0.015     0.530 ± 0.006  0.747 ± 0.008
Sans les PAY_n LogisticRegression            0.751     0.736 - 0.774  15.9 ± 2.4                        57.0 ± 1.3                                             50.0 ± 4.1            6.2 ± 1.0  0.247 ± 0.028     0.432 ± 0.009  0.715 ± 0.004
               MLPClassifier                 0.489     0.455 - 0.517  22.1 ± 4.3                        56.5 ± 1.8                                             52.8 ± 5.2            8.7 ± 1.9  0.315 ± 0.040     0.445 ± 0.005  0.728 ± 0.006
               RandomForest                  0.632     0.629 - 0.636  22.9 ± 0.6                        56.0 ± 1.6                                             52.2 ± 7.1            9.0 ± 0.2  0.325 ± 0.008     0.438 ± 0.007  0.723 ± 0.006
               CatBoost                      0.727     0.704 - 0.744  22.1 ± 3.2                        58.1 ± 0.9                                             54.5 ± 7.4            8.4 ± 1.1  0.319 ± 0.035     0.454 ± 0.005  0.735 ± 0.007

In [17]:
def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

REFERENCE = "Complet"
for mesure in ("Rappel", "Précision moyenne"):
    libelle = "Rappel au seuil optimisé" if mesure == "Rappel" else mesure
    def meilleur(variante):
        return max(MODELES_A_TESTER, key=lambda nom: np.mean(mesures[(variante, nom)][mesure]))
    lignes = []
    for variante in [v for v in VARIANTES if v != REFERENCE]:
        for nom in MODELES_A_TESTER + ["Meilleur contre meilleur"]:
            ref, comp = (meilleur(REFERENCE), meilleur(variante)) if nom == "Meilleur contre meilleur" else (nom, nom)
            score_ref, score_comp = np.array(mesures[(REFERENCE, ref)][mesure]), np.array(mesures[(variante, comp)][mesure])
            gains = score_comp - score_ref
            lignes.append({
                "Variante": variante,
                "Modèle": nom if nom != "Meilleur contre meilleur" else f"{nom} ({ref} → {comp})",
                f"{libelle}, complet": round(score_ref.mean(), 3),
                f"{libelle}, variante": round(score_comp.mean(), 3),
                "Gain": f"{gains.mean():+.3f} ± {gains.std():.3f}",
                "Part du score complet gardée (%)": round(score_comp.mean() / score_ref.mean() * 100, 1),
                "Plis gagnés": f"{(gains > 0).sum()} sur 5",
                "Verdict": verdict(gains),
            })
    display(Markdown(f"**Comparaison avec la variante complète, pli par pli : {libelle}**"))
    display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

**Comparaison avec la variante complète, pli par pli : Rappel au seuil optimisé**

Rappel au seuil optimisé, complet  Rappel au seuil optimisé, variante            Gain  Part du score complet gardée (%) Plis gagnés          Verdict
Variante       Modèle                                                                                                                                                                                                            
PAY_n seuls    LogisticRegression                                                                        0.300                               0.378  +0.078 ± 0.023                             126.0     5 sur 5      ✅ gain réel
               MLPClassifier                                                                             0.355                               0.354  -0.002 ± 0.033                              99.6     3 sur 5  ➖ dans le bruit
               RandomForest                                                                              0.381                               0.355  -0.026 ± 0.024                              93.2     0 sur 5          ❌ perte
               CatBoost                                                                                  0.373                               0.362  -0.010 ± 0.014                              97.3     2 sur 5  ➖ dans le bruit
               Meilleur contre meilleur (RandomForest → LogisticRegression)                              0.381                               0.378  -0.003 ± 0.030                              99.1     3 sur 5  ➖ dans le bruit
Sans les PAY_n LogisticRegression                                                                        0.300                               0.159  -0.141 ± 0.042                              52.9     0 sur 5          ❌ perte
               MLPClassifier                                                                             0.355                               0.221  -0.134 ± 0.035                              62.1     0 sur 5          ❌ perte
               RandomForest                                                                              0.381                               0.229  -0.152 ± 0.037                              60.1     0 sur 5          ❌ perte
               CatBoost                                                                                  0.373                               0.221  -0.152 ± 0.048                              59.3     0 sur 5          ❌ perte
               Meilleur contre meilleur (RandomForest → RandomForest)                                    0.381                               0.229  -0.152 ± 0.037                              60.1     0 sur 5          ❌ perte

**Comparaison avec la variante complète, pli par pli : Précision moyenne**

Précision moyenne, complet  Précision moyenne, variante            Gain  Part du score complet gardée (%) Plis gagnés  Verdict
Variante       Modèle                                                                                                                                                                            
PAY_n seuls    LogisticRegression                                                       0.499                        0.486  -0.013 ± 0.004                              97.3     0 sur 5  ❌ perte
               MLPClassifier                                                            0.540                        0.518  -0.023 ± 0.009                              95.8     0 sur 5  ❌ perte
               RandomForest                                                             0.551                        0.531  -0.021 ± 0.004                              96.3     0 sur 5  ❌ perte
               CatBoost                                                                 0.556                        0.530  -0.026 ± 0.006                              95.3     0 sur 5  ❌ perte
               Meilleur contre meilleur (CatBoost → RandomForest)                       0.556                        0.531  -0.026 ± 0.004                              95.4     0 sur 5  ❌ perte
Sans les PAY_n LogisticRegression                                                       0.499                        0.432  -0.068 ± 0.007                              86.5     0 sur 5  ❌ perte
               MLPClassifier                                                            0.540                        0.445  -0.095 ± 0.009                              82.4     0 sur 5  ❌ perte
               RandomForest                                                             0.551                        0.438  -0.113 ± 0.007                              79.4     0 sur 5  ❌ perte
               CatBoost                                                                 0.556                        0.454  -0.102 ± 0.008                              81.7     0 sur 5  ❌ perte
               Meilleur contre meilleur (CatBoost → CatBoost)                           0.556                        0.454  -0.102 ± 0.008                              81.7     0 sur 5  ❌ perte

**Comment lire** :
- **Détection au seuil optimisé** : le rappel dit combien de défauts chaque modèle détecte quand il ne signale un client que si, à la marge, cela vaut la peine (au plus 1 bon client pour 1 défaut). La précision sur l'ensemble des clients signalés est plus haute que la précision marginale : les clients les plus risqués, signalés en premier, sont presque tous en défaut. La **précision marginale** vérifie la règle : proche de 50 %, le seuil s'arrête au bon endroit ; nettement au-dessus, il s'arrête trop tôt (des défauts rentables restent à aller chercher) ; nettement en dessous, trop tard. Elle est mesurée sur peu de clients, donc bruitée. Un seuil qui varie beaucoup d'un pli à l'autre (colonne « min - max ») signale un compromis instable.
- **Deux comparaisons**, une ligne par modèle et par variante, comparée au même modèle sur les 23 variables, puis le meilleur modèle de chaque variante face au meilleur de la variante complète. La précision moyenne juge la capacité du modèle à classer les défauts au-dessus des bons clients, tous seuils confondus ; le rappel au seuil optimisé juge ce que cette capacité donne avec la règle retenue par le métier.
- **`PAY_n` seuls** : si le verdict est « dans le bruit », les six codifications portent à elles seules ce que les modèles trouvent dans le jeu de base ; la colonne « Part du score complet gardée » mesure ce qu'elles en gardent si elles perdent.
- **Sans les `PAY_n`** : l'écart avec la variante complète mesure ce que le comportement du client, décrit par les montants (plafond, factures, paiements), et la démographie ne savent pas retrouver sans l'étiquette de la banque.
- Les deux variantes se lisent ensemble, pour la problématique : si les `PAY_n` seuls gardent presque tout et que leur retrait coûte beaucoup, le signal du jeu de base tient pour l'essentiel aux codifications ; le comportement brut, tel que les montants le donnent, n'explique alors qu'une partie du défaut. Si les montants seuls restent proches de la variante complète, le comportement de paiement porte bien le défaut, et les codifications n'en sont qu'un résumé.

Rappel : le test du ML est inclus dans ces 30 000 clients. Ces résultats éclairent une hypothèse, ils ne servent pas à choisir le modèle final ni ses variables.

### Comparaison avec l'étude d'origine (Yeh et Lien, 2009)

L'étude d'origine (`docs/DefaultCreditCardClients_yeh_2009.pdf`, traduite dans `docs/traduction_DefaultCreditCardClients_yeh_2009.md`) travaille sur le même fichier brut et compare six méthodes sur **un seul découpage** entraînement / validation. Elle ne publie **ni ROC AUC ni rappel**, mais deux mesures (Tableau 1, p. 2477, en validation) :

| Méthode (Yeh et Lien) | Taux d'erreur | Ratio d'aire |
|---|---|---|
| Réseau de neurones | 0,17 | 0,54 |
| Arbre de classification | 0,17 | 0,536 |
| Bayésien naïf | 0,21 | 0,53 |
| KNN | 0,16 | 0,45 |
| Régression logistique | 0,18 | 0,44 |
| Analyse discriminante | 0,26 | 0,43 |

- **Ratio d'aire** (leur équation 1) : sur la courbe de gain (défauts cumulés en fonction du nombre de clients, triés du plus risqué au moins risqué), l'aire entre la courbe du modèle et la diagonale, divisée par l'aire entre la courbe parfaite et la diagonale. Il mesure la **qualité du tri**, sans seuil ; il vaut exactement **2 × ROC AUC − 1**.
- **Taux d'erreur** : la part de clients mal classés, défauts manqués et fausses alertes confondus. Il ne dit pas combien de défauts sont trouvés : ne signaler personne donne déjà un taux d'erreur égal au taux de défaut. Il se compare à notre **règle marginale à 1 pour 1**, qui compte une fausse alerte autant qu'un défaut manqué et revient donc à chercher le taux d'erreur le plus bas. Il est aussi donné au seuil de 0,5 sur les probabilités, la règle par défaut d'un modèle ; nos modèles pondèrent les défauts (`class_weight="balanced"`, `scale_pos_weight`), leurs probabilités sont donc gonflées et ce seuil signale beaucoup plus de clients.

Le tableau ci-dessous calcule ces mesures, pli par pli, sur les probabilités hors pli de chaque modèle et de chaque variante (moyenne ± écart-type sur les 5 plis).

In [25]:
# Mesures de Yeh et Lien (2009), calculées sur les probabilités hors pli, pli par pli
def ratio_aire(y_vrai, proba):
    # Leur équation 1 : aire entre la courbe de gain du modèle et la diagonale, rapportée à celle de la courbe parfaite
    y_vrai = np.asarray(y_vrai)
    n, d = len(y_vrai), y_vrai.sum()
    cumul = np.cumsum(y_vrai[np.argsort(-proba, kind="mergesort")])   # défauts cumulés, clients du plus risqué au moins risqué
    rang = np.arange(1, n + 1)
    diagonale = d * (n + 1) / 2                                          # somme de la diagonale (tri au hasard)
    return (cumul.sum() - diagonale) / (np.minimum(rang, d).sum() - diagonale)

lignes = []
for (variante, nom), m in mesures.items():
    p = probas[(variante, nom)]
    aire, erreur_regle, erreur_05 = [], [], []
    for k, (_, validation) in enumerate(plis):
        y_val = y[validation]
        aire.append(ratio_aire(y_val, p[validation]))
        # Règle marginale : fausses alertes + défauts manqués, à partir de la détection du pli (section 4)
        erreur_regle.append(m["Signalés"][k] * (1 - m["Précision"][k]) + (1 - m["Rappel"][k]) * y_val.mean())
        erreur_05.append(((p[validation] >= 0.5) != y_val).mean())
    lignes.append({
        "Variante": variante, "Modèle": nom,
        "Ratio d'aire": moy_et(aire),
        "Contrôle : 2 × ROC AUC − 1": f"{2 * np.mean(m['ROC AUC']) - 1:.3f}",
        "Taux d'erreur, règle marginale": moy_et(erreur_regle),
        "Taux d'erreur, seuil 0,5": moy_et(erreur_05),
        "Taux d'erreur si personne n'est signalé": f"{TAUX_DEFAUT:.3f}",
    })
display(Markdown("**Mesures de Yeh et Lien (2009) sur les probabilités hors pli** (moyenne ± écart-type sur les 5 plis)"))
display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

**Mesures de Yeh et Lien (2009) sur les probabilités hors pli** (moyenne ± écart-type sur les 5 plis)

Ratio d'aire Contrôle : 2 × ROC AUC − 1 Taux d'erreur, règle marginale Taux d'erreur, seuil 0,5 Taux d'erreur si personne n'est signalé
Variante       Modèle                                                                                                                                                      
Complet        LogisticRegression  0.499 ± 0.009                      0.499                  0.195 ± 0.002            0.262 ± 0.001                                   0.221
               MLPClassifier       0.549 ± 0.009                      0.549                  0.183 ± 0.004            0.182 ± 0.004                                   0.221
               RandomForest        0.560 ± 0.012                      0.560                  0.185 ± 0.002            0.233 ± 0.003                                   0.221
               CatBoost            0.567 ± 0.011                      0.567                  0.180 ± 0.003            0.237 ± 0.004                                   0.221
PAY_n seuls    LogisticRegression  0.411 ± 0.012                      0.410                  0.180 ± 0.004            0.221 ± 0.004                                   0.221
               MLPClassifier       0.486 ± 0.018                      0.484                  0.180 ± 0.003            0.179 ± 0.004                                   0.221
               RandomForest        0.500 ± 0.011                      0.498                  0.179 ± 0.005            0.218 ± 0.004                                   0.221
               CatBoost            0.496 ± 0.015                      0.494                  0.180 ± 0.004            0.219 ± 0.005                                   0.221
Sans les PAY_n LogisticRegression  0.430 ± 0.008                      0.430                  0.213 ± 0.001            0.327 ± 0.003                                   0.221
               MLPClassifier       0.456 ± 0.012                      0.456                  0.210 ± 0.002            0.210 ± 0.002                                   0.221
               RandomForest        0.445 ± 0.011                      0.445                  0.210 ± 0.003            0.341 ± 0.006                                   0.221
               CatBoost            0.470 ± 0.013                      0.470                  0.207 ± 0.003            0.313 ± 0.002                                   0.221

**Comment lire** : seule la variante complète se compare à l'étude d'origine, qui utilise les 23 variables.
- **Ratio d'aire** : plus il est haut, mieux le modèle trie les défauts avant les bons clients. Un ratio d'aire au-dessus de celui de leur meilleur modèle (le réseau de neurones) veut dire un meilleur tri, sur la même population.
- **Taux d'erreur à la règle marginale** : sous le leur, nos modèles se trompent moins souvent, fausses alertes et défauts manqués confondus ; au même niveau, ils ne font pas mieux sur ce terrain. Le taux d'erreur ne dit pas combien de défauts sont détectés : le rappel est dans le tableau « Détection au seuil optimisé ».
- **Prudence** : leurs chiffres viennent d'un seul découpage et sont arrondis au centième ; les nôtres sont des moyennes sur 5 plis. L'article contient aussi des incohérences : il annonce 25 000 observations (le fichier en compte 30 000) et 22,12 % de défauts puis 87,88 % de clients sans risque ; ses courbes de gain (environ 15 000 clients en validation, dont environ 3 300 défauts) confirment bien un taux de défaut d'environ 22 %. Un écart de quelques centièmes ne se tranche pas.

## 5. Ce qui porte les prédictions : importance par permutation

**Principe** : sur chaque pli de validation, on reprend le modèle entraîné sur les 4 autres plis (section 4) et on **mélange** les valeurs d'une famille de colonnes entre les clients (les autres colonnes restent intactes). Le modèle perd ce que cette famille lui apportait : la **perte de précision moyenne** mesure son importance. La précision moyenne ne dépend d'aucun seuil : la mesure juge la capacité de détection, pas un point de fonctionnement, et elle est plus stable qu'un score pris en un seul point. Le mélange est répété 5 fois (graine fixée), puis la perte est moyennée sur les répétitions et sur les 5 plis. Une famille se mélange d'un bloc (les six mois ensemble, avec le même tirage pour chaque colonne), pour garder sa cohérence d'un mois à l'autre.

**Familles** : codifications (`PAY_1` à `PAY_6`), factures (`BILL_AMT1` à `6`), paiements (`PAY_AMT1` à `6`), plafond (`LIMIT_BAL`), démographie (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`). Le comportement du client, au sens des montants, ce sont les factures, les paiements et le plafond ; les codifications sont l'étiquette posée par la banque.

**Deux tableaux** :
- **par famille**, pour chaque variante et chaque modèle : perte de précision moyenne et son écart-type sur les plis, et la part de chaque famille dans la perte totale de la colonne ;
- **par colonne**, mélangée seule : le détail, pour voir quels mois et quels montants comptent.

**Ce que la permutation mesure** : ce que le modèle perd **quand la famille manque, les autres étant là**. Deux familles qui disent la même chose (les codifications et les paiements, par exemple) se remplacent l'une l'autre : chacune paraît alors peu importante, alors qu'ensemble elles portent beaucoup. C'est pourquoi cette section se lit avec la section 4 : les variantes « `PAY_n` seuls » et « Sans les `PAY_n` » disent ce que chaque bloc porte **seul**, la permutation dit ce qu'il apporte **en plus des autres**.

In [18]:
FAMILLES = {
    "Codifications (PAY_n)": PAY,
    "Factures (BILL_AMTn)": [f"BILL_AMT{n}" for n in range(1, 7)],
    "Paiements (PAY_AMTn)": [f"PAY_AMT{n}" for n in range(1, 7)],
    "Plafond (LIMIT_BAL)": ["LIMIT_BAL"],
    "Démographie": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
REPETITIONS = 5
generateur = np.random.default_rng(42)

def perte_par_permutation(modeles, X, colonnes):
    # Perte de précision moyenne sur chaque pli de validation quand les colonnes sont mélangées d'un bloc (même tirage pour toutes),
    # moyennée sur les répétitions
    pertes = []
    for m, (_, validation) in zip(modeles, plis):
        X_val, y_val = X.iloc[validation], y[validation]
        reference = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        perte = []
        for _ in range(REPETITIONS):
            melange = X_val.copy()
            ordre_melange = generateur.permutation(len(melange))
            melange[colonnes] = melange[colonnes].to_numpy()[ordre_melange]
            perte.append(reference - average_precision_score(y_val, m.predict_proba(melange)[:, 1]))
        pertes.append(np.mean(perte))
    return pertes

importance_familles, importance_colonnes = {}, {}
for (variante, nom), modeles in modeles_plis.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for famille, colonnes in FAMILLES.items():
        presentes = [c for c in colonnes if c in X.columns]
        if presentes:
            importance_familles[(variante, nom, famille)] = perte_par_permutation(modeles, X, presentes)
    for colonne in X.columns:
        importance_colonnes[(variante, nom, colonne)] = perte_par_permutation(modeles, X, [colonne])
    print(f"{variante}, {nom} : fait")

Complet, LogisticRegression : fait
Complet, MLPClassifier : fait
Complet, RandomForest : fait
Complet, CatBoost : fait
PAY_n seuls, LogisticRegression : fait
PAY_n seuls, MLPClassifier : fait
PAY_n seuls, RandomForest : fait
PAY_n seuls, CatBoost : fait
Sans les PAY_n, LogisticRegression : fait
Sans les PAY_n, MLPClassifier : fait
Sans les PAY_n, RandomForest : fait
Sans les PAY_n, CatBoost : fait


In [19]:
# Par famille : une colonne par variante et par modèle, une ligne par famille
colonnes_tableau = list(dict.fromkeys((variante, nom) for variante, nom, _ in importance_familles))
perte = pd.DataFrame({cle: {famille: np.mean(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
ecart = pd.DataFrame({cle: {famille: np.std(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
part = perte.clip(lower=0) / perte.clip(lower=0).sum() * 100

display(Markdown("**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)"))
texte = (perte.round(3).astype(str) + " ± " + ecart.round(3).astype(str)).where(perte.notna(), "—")
display(texte.rename_axis(index="Famille", columns=["Variante", "Modèle"]))
display(Markdown("**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)"))
display(part.round(1).rename_axis(index="Famille", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)

Variante                         Complet                                                     PAY_n seuls                                                 Sans les PAY_n                                             
Modèle                LogisticRegression  MLPClassifier   RandomForest       CatBoost LogisticRegression  MLPClassifier   RandomForest      CatBoost LogisticRegression  MLPClassifier   RandomForest       CatBoost
Famille                                                                                                                                                                                                             
Codifications (PAY_n)      0.219 ± 0.006   0.268 ± 0.01  0.292 ± 0.007   0.29 ± 0.008      0.264 ± 0.008  0.295 ± 0.004  0.308 ± 0.008  0.31 ± 0.005                  —              —              —              —
Factures (BILL_AMTn)       0.001 ± 0.002  0.025 ± 0.003  0.014 ± 0.002  0.018 ± 0.003                  —              —              —             —      0.119 ± 0.008  0.129 ± 0.009  0.098 ± 0.011  0.137 ± 0.007
Paiements (PAY_AMTn)       0.079 ± 0.005  0.046 ± 0.006  0.028 ± 0.003  0.029 ± 0.004                  —              —              —             —       0.198 ± 0.01  0.209 ± 0.006  0.187 ± 0.008  0.213 ± 0.006
Plafond (LIMIT_BAL)        0.004 ± 0.002  0.006 ± 0.001  0.005 ± 0.001  0.011 ± 0.002                  —              —              —             —      0.014 ± 0.002   0.02 ± 0.005  0.032 ± 0.006  0.032 ± 0.005
Démographie                0.007 ± 0.001  0.006 ± 0.002  0.002 ± 0.001  0.005 ± 0.001                  —              —              —             —      0.009 ± 0.002  0.008 ± 0.002  0.003 ± 0.001  0.008 ± 0.002

**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)

Variante                         Complet                                            PAY_n seuls                                         Sans les PAY_n                                    
Modèle                LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Famille                                                                                                                                                                                   
Codifications (PAY_n)               70.7          76.2         85.7     82.0              100.0         100.0        100.0    100.0                NaN           NaN          NaN      NaN
Factures (BILL_AMTn)                 0.2           7.2          4.0      5.2                NaN           NaN          NaN      NaN               35.0          35.2         30.7     35.2
Paiements (PAY_AMTn)                25.6          13.2          8.2      8.2                NaN           NaN          NaN      NaN               58.4          57.2         58.4     54.7
Plafond (LIMIT_BAL)                  1.2           1.8          1.5      3.2                NaN           NaN          NaN      NaN                4.0           5.4         10.0      8.1
Démographie                          2.3           1.6          0.5      1.5                NaN           NaN          NaN      NaN                2.6           2.2          1.0      1.9

In [20]:
# Par colonne, mélangée seule : perte de précision moyenne sur les 5 plis, triée sur la variante complète (moyenne des 4 modèles)
par_colonne = pd.DataFrame({(variante, nom): {colonne: np.mean(importance_colonnes[(variante, nom, colonne)])
                                              for v, n, colonne in importance_colonnes if (v, n) == (variante, nom)}
                            for variante, nom in colonnes_tableau})
ordre_colonnes = par_colonne["Complet"].mean(axis=1).sort_values(ascending=False).index
with pd.option_context("display.max_rows", None):
    display(Markdown("**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)"))
    display(par_colonne.loc[ordre_colonnes].round(3).rename_axis(index="Colonne", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)

Variante             Complet                                            PAY_n seuls                                         Sans les PAY_n                                    
Modèle    LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Colonne                                                                                                                                                                       
PAY_1                  0.102         0.207        0.133    0.190              0.216         0.206        0.180    0.196                NaN           NaN          NaN      NaN
PAY_2                  0.014         0.006        0.013    0.004              0.002         0.018        0.019    0.014                NaN           NaN          NaN      NaN
PAY_5                  0.015         0.010        0.004    0.003             -0.003         0.012        0.002    0.005                NaN           NaN          NaN      NaN
PAY_6                  0.011         0.010        0.004    0.005             -0.005         0.011        0.003    0.013                NaN           NaN          NaN      NaN
LIMIT_BAL              0.004         0.007        0.005    0.011                NaN           NaN          NaN      NaN              0.012         0.019        0.030    0.033
PAY_3                  0.008         0.005        0.006    0.005              0.004         0.009        0.007    0.010                NaN           NaN          NaN      NaN
PAY_AMT2               0.005         0.010        0.002    0.003                NaN           NaN          NaN      NaN              0.058         0.075        0.051    0.069
BILL_AMT1             -0.003         0.011        0.004    0.008                NaN           NaN          NaN      NaN              0.001         0.012        0.006    0.015
PAY_AMT1               0.001         0.009        0.004    0.002                NaN           NaN          NaN      NaN              0.068         0.077        0.082    0.074
PAY_AMT3               0.003         0.006        0.002    0.003                NaN           NaN          NaN      NaN              0.030         0.040        0.031    0.041
PAY_4                  0.001         0.005        0.005    0.003             -0.003         0.012        0.005    0.007                NaN           NaN          NaN      NaN
BILL_AMT3              0.008         0.003        0.002    0.001                NaN           NaN          NaN      NaN              0.038         0.025        0.010    0.022
BILL_AMT2             -0.001         0.009        0.002    0.001                NaN           NaN          NaN      NaN              0.037         0.022        0.013    0.023
BILL_AMT4              0.003         0.004        0.001    0.000                NaN           NaN          NaN      NaN              0.027         0.021        0.009    0.015
EDUCATION              0.003         0.001        0.001    0.003                NaN           NaN          NaN      NaN              0.003         0.003        0.001    0.002
BILL_AMT6              0.004         0.001        0.001    0.000                NaN           NaN          NaN      NaN              0.031         0.017        0.010    0.016
MARRIAGE               0.002         0.002        0.000    0.002                NaN           NaN          NaN      NaN              0.003         0.004        0.001    0.003
PAY_AMT4              -0.001         0.002        0.002    0.001                NaN           NaN          NaN      NaN              0.013         0.032        0.020    0.025
BILL_AMT5             -0.001         0.003        0.001    0.001                NaN           NaN          NaN      NaN              0.007         0.010        0.008    0.005
SEX                    0.001         0.001        0.000    0.001                NaN           NaN          NaN      NaN              0.002         0.0

**Comment lire** :
- **Par famille** : plus la perte est grande, plus le modèle s'appuie sur la famille. Une perte plus petite que son écart-type est dans le bruit : la famille n'apporte rien de plus que les autres. Si, dans la variante complète, les codifications portent l'essentiel de la perte, les modèles s'appuient d'abord sur l'étiquette de la banque ; si ce sont les factures et les paiements, ils s'appuient sur le comportement de paiement du client.
- **Par colonne** : une colonne mélangée seule perd moins que sa famille, puisque les autres mois la remplacent en partie. Le tableau sert surtout à voir **quel mois** compte (`PAY_1`, le plus récent, ou l'historique) et **quel montant**.
- **Avec la section 4** : une famille peu importante ici, mais qui garde l'essentiel du score seule (variante « `PAY_n` seuls ») ou sans laquelle le score tient (variante « Sans les `PAY_n` »), dit la même chose que les autres familles : l'information est **redondante**, pas absente.

## 6. Comparaison avec le travail mené dans les scénarios

**Ce qui est comparé** : les modèles du jeu de base (23 variables brutes, niveau 0) face aux scénarios du ML, qui reposent sur le nettoyage jusqu'au niveau 5, le retrait des clients au contentieux à M, les `PAY_n` corrigés et les colonnes construites. Scénarios comparés (à choisir dans la cellule) : `ml_0`, le socle (23 variables d'origine, mais nettoyées et corrigées), et `ml_4`, la référence retenue.

**Deux lectures** :
1. **Sur les mêmes clients : la seule comparaison juste.** Les scénarios n'ont vu que les clients du train ML ; le jeu de base en a 30 000, dont les clients au contentieux, très faciles à repérer (leurs `PAY_n` restent à 2 et plus) et qui gonflent ses scores. On ne garde donc, des probabilités hors pli du jeu de base, que les clients du train ML, découpés selon **les 5 plis des scénarios** ; sur chaque pli, le seuil du rappel minimal et le score décisionnel F2 sont recalculés comme dans les scénarios. Chaque client reste prédit par un modèle qui ne l'a pas vu. Les modèles du jeu de base ont cependant appris sur d'autres clients (contentieux et test du ML compris), avec des `PAY_n` non corrigés : c'est précisément ce qu'on veut mesurer.
2. **Chacun sur sa population : à titre indicatif.** Les scores tels que chaque notebook les obtient. Le taux de défaut n'est pas le même (le contentieux en porte une grande part dans le jeu de base) : la précision est donc aussi donnée **rapportée au taux de défaut** (combien de fois mieux que le hasard), mesure à peu près comparable d'une population à l'autre.

**Mesure** : pour rester comparable aux scénarios, cette section reprend leur mesure, le **score décisionnel F2 au rappel minimal de 60 %**, calculée sur les probabilités hors pli du jeu de base, et non le F2 au seuil optimisé de la section 4.

Verdict des scénarios, pli par pli : « gain réel » si le gain moyen du jeu de base dépasse l'écart-type des gains, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [21]:
# ==============================================================================
# SCÉNARIOS COMPARÉS : numéros des scénarios dont les probabilités hors pli (méthode v8) sont lues
# ==============================================================================
SCENARIOS_COMPARES = ["0", "4"]   # ml_0 : socle ; ml_4 : référence retenue
VERSION_METHODE = "v8"

dossier_probas = root_dir / "data" / "ML" / "probas_hors_pli"
train_ml = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv", usecols=["ID", "dpnm"])
y_ml = train_ml["dpnm"].to_numpy()
plis_ml = list(cv.split(train_ml, train_ml["dpnm"]))   # les 5 plis des scénarios (même ordre des clients, même graine)

# Probabilités hors pli du jeu de base, restreintes aux clients du train ML, dans l'ordre du train ML
position = pd.Series(np.arange(len(original)), index=original["ID"])
assert train_ml["ID"].isin(position.index).all(), "Des clients du train ML manquent dans le jeu de base"
lignes_ml = position.loc[train_ml["ID"]].to_numpy()
assert (y[lignes_ml] == y_ml).all(), "Cible différente entre le jeu de base et le train ML"

f2_ml = {}   # (source, modèle) -> F2 sur les 5 plis des scénarios
for (variante, nom), p in probas.items():
    p_ml = p[lignes_ml]
    f2_ml[(f"Jeu de base, {variante}", nom)] = [f2_au_rappel_minimal(y_ml[val], p_ml[val]) for _, val in plis_ml]

for numero in SCENARIOS_COMPARES:
    ref = pd.read_csv(dossier_probas / f"ml_{numero}_{VERSION_METHODE}.csv").set_index("ID").loc[train_ml["ID"]]
    assert (ref["dpnm"].to_numpy() == y_ml).all(), f"Probabilités de ml_{numero} non alignées sur le train ML"
    for nom in [m for m in MODELES_A_TESTER if m in ref.columns]:
        f2_ml[(f"ml_{numero}", nom)] = [f2_au_rappel_minimal(y_ml[val], ref[nom].to_numpy()[val]) for _, val in plis_ml]

sources = list(dict.fromkeys(source for source, _ in f2_ml))
tableau = pd.DataFrame({source: {nom: f"{np.mean(f2_ml[(source, nom)]):.3f} ± {np.std(f2_ml[(source, nom)]):.3f}"
                                 for nom in MODELES_A_TESTER if (source, nom) in f2_ml} for source in sources}).T
display(Markdown(f"**Lecture 1 : sur les {len(train_ml)} clients du train ML, plis des scénarios** (score décisionnel F2 ± écart-type sur les plis ; "
                 f"taux de défaut {y_ml.mean() * 100:.1f} %)"))
display(tableau.rename_axis("Source"))

**Lecture 1 : sur les 19358 clients du train ML, plis des scénarios** (score décisionnel F2 ± écart-type sur les plis ; taux de défaut 15.9 %)

,LogisticRegression,MLPClassifier,RandomForest,CatBoost
Source,,,,
"Jeu de base, Complet",0.467 ± 0.005,0.481 ± 0.011,0.483 ± 0.009,0.485 ± 0.010
"Jeu de base, PAY_n seuls",0.405 ± 0.008,0.452 ± 0.010,0.455 ± 0.010,0.455 ± 0.014
"Jeu de base, Sans les PAY_n",0.468 ± 0.006,0.475 ± 0.008,0.471 ± 0.008,0.482 ± 0.010
ml_0,0.482 ± 0.010,0.477 ± 0.009,0.479 ± 0.010,0.482 ± 0.010
ml_4,0.482 ± 0.009,0.476 ± 0.010,0.480 ± 0.008,0.484 ± 0.010


In [22]:
def meilleur_de(source):
    return max([nom for s, nom in f2_ml if s == source], key=lambda nom: np.mean(f2_ml[(source, nom)]))

lignes = []
for variante in VARIANTES:
    source = f"Jeu de base, {variante}"
    for numero in SCENARIOS_COMPARES:
        scenario = f"ml_{numero}"
        paires = [(nom, nom) for nom in MODELES_A_TESTER if (scenario, nom) in f2_ml] + [(meilleur_de(scenario), meilleur_de(source))]
        for i, (nom_ref, nom_base) in enumerate(paires):
            gains = np.array(f2_ml[(source, nom_base)]) - np.array(f2_ml[(scenario, nom_ref)])
            lignes.append({
                "Jeu de base": variante,
                "Face à": scenario,
                "Modèle": nom_ref if i < len(paires) - 1 else f"Meilleur contre meilleur ({nom_ref} → {nom_base})",
                "F2 scénario": round(np.mean(f2_ml[(scenario, nom_ref)]), 3),
                "F2 jeu de base": round(np.mean(f2_ml[(source, nom_base)]), 3),
                "Gain du jeu de base": f"{gains.mean():+.3f} ± {gains.std():.3f}",
                "Plis gagnés": f"{(gains > 0).sum()} sur 5",
                "Verdict": verdict(gains),
            })
display(Markdown("**Lecture 1 : le jeu de base face aux scénarios, pli par pli, sur les mêmes clients**"))
display(pd.DataFrame(lignes).set_index(["Jeu de base", "Face à", "Modèle"]))

**Lecture 1 : le jeu de base face aux scénarios, pli par pli, sur les mêmes clients**

F2 scénario  F2 jeu de base Gain du jeu de base Plis gagnés          Verdict
Jeu de base    Face à Modèle                                                                                                                          
Complet        ml_0   LogisticRegression                                        0.482           0.467      -0.014 ± 0.005     0 sur 5          ❌ perte
                      MLPClassifier                                             0.477           0.481      +0.005 ± 0.005     4 sur 5  ➖ dans le bruit
                      RandomForest                                              0.479           0.483      +0.004 ± 0.003     5 sur 5      ✅ gain réel
                      CatBoost                                                  0.482           0.485      +0.003 ± 0.003     4 sur 5  ➖ dans le bruit
                      Meilleur contre meilleur (CatBoost → CatBoost)            0.482           0.485      +0.003 ± 0.003     4 sur 5  ➖ dans le bruit
               ml_4   LogisticRegression                                        0.482           0.467      -0.015 ± 0.005     0 sur 5          ❌ perte
                      MLPClassifier                                             0.476           0.481      +0.006 ± 0.006     3 sur 5  ➖ dans le bruit
                      RandomForest                                              0.480           0.483      +0.003 ± 0.003     4 sur 5  ➖ dans le bruit
                      CatBoost                                                  0.484           0.485      +0.001 ± 0.004     3 sur 5  ➖ dans le bruit
                      Meilleur contre meilleur (CatBoost → CatBoost)            0.484           0.485      +0.001 ± 0.004     3 sur 5  ➖ dans le bruit
PAY_n seuls    ml_0   LogisticRegression                                        0.482           0.405      -0.076 ± 0.014     0 sur 5          ❌ perte
                      MLPClassifier                                             0.477           0.452      -0.024 ± 0.008     0 sur 5          ❌ perte
                      RandomForest                                              0.479           0.455      -0.024 ± 0.011     0 sur 5          ❌ perte
                      CatBoost                                                  0.482           0.455      -0.027 ± 0.011     0 sur 5          ❌ perte
                      Meilleur contre meilleur (CatBoost → RandomForest)        0.482           0.455      -0.027 ± 0.010     0 sur 5          ❌ perte
               ml_4   LogisticRegression                                        0.482           0.405      -0.077 ± 0.013     0 sur 5          ❌ perte
                      MLPClassifier                                             0.476           0.452      -0.023 ± 0.005     0 sur 5          ❌ perte
                      RandomForest                                              0.480           0.455      -0.024 ± 0.011     0 sur 5          ❌ perte
                      CatBoost                                                  0.484           0.455      -0.029 ± 0.012     0 sur 5          ❌ perte
                      Meilleur contre meilleur (CatBoost → RandomForest)        0.484           0.455      -0.029 ± 0.011     0 sur 5          ❌ perte
Sans les PAY_n ml_0   LogisticRegression                                        0.482           0.468      -0.014 ± 0.005     0 sur 5          ❌ perte
                      MLPClassifier                                             0.477           0.475      -0.001 ± 0.003     2 sur 5  ➖ dans le bruit
                      RandomForest                                              0.479           0.471      -0.009 ± 0.003     0 sur 5          ❌ perte
                      CatBoost                                                  0.482           0.482      -0.000 ± 0.004     3 sur 5  ➖ dans le bruit
                      Meilleur contre meilleur (CatBoost → CatBoost)            0.482           0.482      -0.000 ± 0.004     3 sur 5  ➖ dans le bru

In [24]:
# Lecture 2 : chacun sur sa population (résultats des scénarios lus dans le fichier de résultats, méthode v8)
resultats_scenarios = pd.read_csv(root_dir / "data" / "ML" / "resultats_scenarios.csv")
lignes = []
for (variante, nom), p in probas.items():
    signales = p >= seuil_pour_rappel(y, p)
    lignes.append({"Source": f"Jeu de base, {variante}", "Modèle": nom, "Clients": len(y), "Taux de défaut (%)": round(TAUX_DEFAUT * 100, 1),
                   "Score décisionnel F2": round(np.mean([f2_au_rappel_minimal(y[val], p[val]) for _, val in plis]), 3),
                   f"{LIBELLE_SCORE} (%)": round(y[signales].mean() * 100, 1),
                   "Précision / taux de défaut": round(y[signales].mean() / TAUX_DEFAUT, 2)})
for numero in SCENARIOS_COMPARES:
    r = resultats_scenarios[(resultats_scenarios["scenario"] == f"ml_{numero}") & (resultats_scenarios["version_methode"] == VERSION_METHODE)]
    for _, ligne in r[r["modele"].isin(MODELES_A_TESTER)].iterrows():
        lignes.append({"Source": f"ml_{numero}", "Modèle": ligne["modele"], "Clients": len(y_ml), "Taux de défaut (%)": round(y_ml.mean() * 100, 1),
                       "Score décisionnel F2": round(ligne["f2_val"], 3), f"{LIBELLE_SCORE} (%)": round(ligne["precision_val"] * 100, 1),
                       "Précision / taux de défaut": round(ligne["precision_val"] / y_ml.mean(), 2)})
display(Markdown("**Lecture 2 : chacun sur sa population, à titre indicatif** (populations et taux de défaut différents)"))
display(pd.DataFrame(lignes).set_index(["Source", "Modèle"]))

**Lecture 2 : chacun sur sa population, à titre indicatif** (populations et taux de défaut différents)

Clients  Taux de défaut (%)  Score décisionnel F2  Précision des défauts prédits (%)  Précision / taux de défaut
Source                      Modèle                                                                                                                              
Jeu de base, Complet        LogisticRegression    30000                22.1                 0.564                               45.5                        2.06
                            MLPClassifier         30000                22.1                 0.574                               48.9                        2.21
                            RandomForest          30000                22.1                 0.576                               49.5                        2.24
                            CatBoost              30000                22.1                 0.578                               50.4                        2.28
Jeu de base, PAY_n seuls    LogisticRegression    30000                22.1                 0.553                               37.5                        1.70
                            MLPClassifier         30000                22.1                 0.568                               45.8                        2.07
                            RandomForest          30000                22.1                 0.572                               47.6                        2.15
                            CatBoost              30000                22.1                 0.573                               47.6                        2.15
Jeu de base, Sans les PAY_n LogisticRegression    30000                22.1                 0.538                               37.9                        1.71
                            MLPClassifier         30000                22.1                 0.542                               38.6                        1.74
                            RandomForest          30000                22.1                 0.537                               37.9                        1.71
                            CatBoost              30000                22.1                 0.546                               40.1                        1.81
ml_0                        CatBoost              19358                15.9                 0.482                               26.9                        1.69
                            LogisticRegression    19358                15.9                 0.482                               26.9                        1.69
                            RandomForest          19358                15.9                 0.479                               26.5                        1.66
                            MLPClassifier         19358                15.9                 0.477                               26.1                        1.64
ml_4                        CatBoost              19358                15.9                 0.484                               27.3                        1.71
                            LogisticRegression    19358                15.9                 0.482                               27.0                        1.69
                            RandomForest          19358                15.9                 0.480                               26.5                        1.67
                            MLPClassifier         19358                15.9                 0.476                               26.0                        1.63

**Comment lire** :
- **Lecture 1**, la seule qui tranche. Si le jeu de base complet est **dans le bruit** face à `ml_4`, le nettoyage, les corrections des `PAY_n` et les colonnes construites n'ont rien apporté à la prédiction sur ces clients : leur apport est ailleurs (lisibilité, justification métier, retrait du contentieux). Si `ml_4` gagne, le travail apporte un gain mesurable. Les lignes « `PAY_n` seuls » et « Sans les `PAY_n` » situent chaque famille de colonnes du jeu de base face au travail mené.
- **Lecture 2** : les scores du jeu de base sur ses 30 000 clients sont attendus plus hauts, car le contentieux y est facile à repérer ; c'est pourquoi ils ne se comparent pas directement aux scénarios. Le rapport précision / taux de défaut corrige en partie l'écart de taux de défaut, pas l'écart de population.

Rappel : le test du ML est inclus dans le jeu de base. Ces comparaisons éclairent la problématique, elles ne servent pas à choisir le modèle final.

## 7. Conclusion : ce que porte le jeu de base

**1. Sur les 30 000 clients, les codifications dominent, mais c'est surtout le contentieux qu'elles portent** (sections 4 et 5). Dans la variante complète, les `PAY_n` portent l'essentiel de la perte quand on les mélange, et `PAY_1` à elle seule pèse plus que toutes les autres colonnes. Sans les `PAY_n`, les paiements puis les factures prennent le relais, mais la précision moyenne et le rappel au seuil optimisé baissent nettement ; avec les `PAY_n` seuls, la perte est faible. Ces importances sont mesurées **contentieux compris** : les clients au contentieux, qui font très souvent défaut, se repèrent à leurs `PAY_n` restés à 2 et plus. Elles disent donc d'abord que les codifications repèrent le contentieux, pas ce qui prédit le défaut des autres clients.

**2. Hors contentieux, le comportement du client porte le signal** (section 6, lecture 1 : clients du train ML, plis des scénarios). Sans aucune codification, les montants (factures, paiements, plafond) et la démographie font **jeu égal** avec le meilleur scénario `ml_4` (dans le bruit), alors que les `PAY_n` seuls **perdent** nettement face à lui, sur tous les plis. Une fois le contentieux mis à part, le comportement de paiement, tel que les montants le décrivent, explique le défaut au moins autant que l'étiquette posée par la banque.

**Réponse partielle à la problématique** : oui, le comportement du client explique une partie du défaut, et cette partie ne tient pas aux codifications de la banque. Elle reste modeste : au rappel de 60 %, environ un client signalé sur quatre fait réellement défaut (lecture 2 et scénarios).

**3. Constats annexes**
- Le jeu de base complet, sans nettoyage ni colonne construite, atteint lui aussi le plateau des scénarios (dans le bruit face à `ml_4`) : le travail de préparation n'a pas fait monter le score. Son apport est ailleurs (contentieux retiré, règles métier justifiées, lisibilité). Seule la régression logistique en profite nettement : elle perd sur le jeu brut face à sa version des scénarios.
- **Face à l'étude d'origine** (Yeh et Lien, 2009, fin de la section 4) : elle ne publie ni ROC AUC ni rappel, on ne peut donc pas dire si nos modèles détectent plus de défauts. Sur ses propres mesures, le ratio d'aire (la qualité du tri, égal à 2 × ROC AUC − 1) et le taux d'erreur, la comparaison se lit dans le tableau dédié, avec les réserves indiquées (un seul découpage, chiffres arrondis, incohérences de l'article).
- La règle marginale tient son objectif : la précision mesurée sur les clients signalés les plus proches du seuil tourne autour de 50 % (section 4).

**Réserves** : les modèles du jeu de base ont appris avec le contentieux et les clients du test du ML, sur des `PAY_n` non corrigés ; « dans le bruit » veut dire qu'on ne peut pas départager, pas que les scores sont identiques. Ces résultats éclairent la problématique, ils ne servent pas à choisir le modèle final.